# 01 — EDA: NHTSA complaints (AutoVoice)
Run after `make data`. All numbers below are produced by the code in this notebook from `data/processed/complaints.parquet` — nothing is hardcoded.

In [ ]:
import pandas as pd, json
from pathlib import Path

df = pd.read_parquet('../data/processed/complaints.parquet')
quality = json.loads(Path('../reports/data_quality.json').read_text())
print(f'rows={len(df)}  cols={list(df.columns)}')

## Narrative length distribution

In [ ]:
df['n_chars'] = df['narrative'].str.len()
df['n_words'] = df['narrative'].str.split().str.len()
df[['n_chars','n_words']].describe()

In [ ]:
df['n_chars'].hist(bins=40)
_ = plt.xlabel('chars'); plt.ylabel('count') if False else None

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(10,3))
df['n_chars'].hist(bins=40, ax=ax[0]); ax[0].set_title('Narrative length (chars)')
df['n_words'].hist(bins=40, ax=ax[1]); ax[1].set_title('Narrative length (words)')

## Top components and makes

In [ ]:
df['component'].value_counts().head(15)

In [ ]:
df['make'].value_counts().head(15)

## Indicator ('severity proxy') balance — rule-derived labels come from these

In [ ]:
bal = df[['crash','fire','injured','deaths']].sum()
bal.plot(kind='bar', title='Positive indicator counts')
print(bal.to_dict())

## Missing data & sample narratives

In [ ]:
(df.isna().sum() / len(df)).sort_values(ascending=False)

In [ ]:
for i in range(3):
    r = df.iloc[i]
    print(f"--- {r.make} {r.model} {r.model_year} | {r.component} ---\n{r.narrative[:300]}\n")

## Data-quality notes (written by reports/data_quality.json)

In [ ]:
json.dumps(quality, indent=2)

In [ ]:
Path('../reports/eda_summary.json').write_text(json.dumps({
    'rows': int(len(df)),
    'mean_n_chars': float(df['n_chars'].mean()),
    'median_n_words': float(df['n_words'].median()),
    'top_components': df['component'].value_counts().head(10).to_dict(),
    'indicator_counts': df[['crash','fire','injured','deaths']].sum().astype(int).to_dict(),
}, indent=2))